# RUN THIS ON COLAB'S T4 GPU FOR ~3 MIN RUNTIME

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install faiss-gpu-cu12

# BOUNDING BOX SIM CHECKER

In [ ]:
# CONFIG
ZIP_PATH = "MyFigureCollection.zip" # CHANGE TO ORIGINAL IMAGE ZIP FOLDER
XML_PATH = "annotations.xml" # CHANGE TO CORRESPONDING ANNOTATION.XML FILE
SIM_THRESHOLD = 0.99 # START AT 99% TO FIND NEAR EXACT MATCHES
BATCH_SIZE = 128
K_NEIGHBORS = 10
CROP_SIZE = 224

EMB_FILE = "clip_embeddings.npy"

# LOAD TORCH/CLIP
import clip, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"USING DEVICE: {DEVICE}")
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()

# PARSE CVAT XML image_id mapping)
import xml.etree.ElementTree as ET
tree = ET.parse(XML_PATH)
root = tree.getroot()

boxes = []  # (image_path, (xtl,ytl,xbr,ybr), image_id)
box_areas = {}

for image in root.findall("image"):
    img_id = image.attrib["id"]
    img_name = image.attrib["name"]
    for box in image.findall("box"):
        xtl = float(box.attrib["xtl"])
        ytl = float(box.attrib["ytl"])
        xbr = float(box.attrib["xbr"])
        ybr = float(box.attrib["ybr"])
        boxes.append((img_name, (xtl, ytl, xbr, ybr), img_id))
        box_areas[img_name] = (xbr - xtl) * (ybr - ytl)

print(f"Total bounding boxes: {len(boxes)}")

# OPEN ZIP
import zipfile
zipf = zipfile.ZipFile(ZIP_PATH)

# COMPUTE OR LOAD EMBEDDINGS
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
import gc
import os

box_keys = []

if os.path.exists(EMB_FILE):
    embeddings = np.load(EMB_FILE)
    print(f"Loaded embeddings: {embeddings.shape}")

    # Re-generate box_keys in memory from boxes list
    box_keys = [img_path for img_path, _, _ in boxes]

else:
    embeddings_list = []
    box_keys = []

    for i in tqdm(range(0, len(boxes), BATCH_SIZE), desc="Computing embeddings"):
        batch_boxes = boxes[i:i+BATCH_SIZE]
        batch_tensors = []

        for img_path, (xtl, ytl, xbr, ybr), _ in batch_boxes:
            try:
                with zipf.open(img_path) as f:
                    img = Image.open(f).convert("RGB")
                    crop = img.crop((xtl, ytl, xbr, ybr))
                    batch_tensors.append(preprocess(crop).unsqueeze(0))
                    box_keys.append(img_path)
            except KeyError:
                print(f"Image {img_path} not found in zip.")

        if batch_tensors:
            batch_tensors = torch.cat(batch_tensors).to(DEVICE)
            with torch.no_grad():
                batch_emb = model.encode_image(batch_tensors)
                batch_emb = batch_emb / batch_emb.norm(dim=-1, keepdim=True)
                embeddings_list.append(batch_emb.cpu().numpy())

            del batch_tensors, batch_emb
            torch.cuda.empty_cache()
            gc.collect()

    embeddings = np.vstack(embeddings_list).astype('float32')
    np.save(EMB_FILE, embeddings)
    print(f"Saved embeddings: {embeddings.shape}")

# BUILD FAISS INDEX
import faiss

faiss.normalize_L2(embeddings)
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
print("Faiss index built.")

# DEDUPLICATION
duplicates = []  # list of tuples: (box1, box2, image_id1, image_id2, similarity)

D, I = index.search(embeddings, K_NEIGHBORS)
for i, neighbors in enumerate(tqdm(I, desc="Finding duplicates")):
    anchor = box_keys[i]
    anchor_img_id = boxes[i][2]
    for j_idx, sim in zip(neighbors[1:], D[i,1:]):  # skip self
        neighbor = box_keys[j_idx]
        neighbor_img_id = boxes[j_idx][2]
        if sim >= SIM_THRESHOLD:
            # symmetric logging: only log once
            if (neighbor, anchor) not in [(b1,b2) for b1,b2,_,_,_ in duplicates]:
                duplicates.append((anchor, neighbor, anchor_img_id, neighbor_img_id, sim))

print(f"Total near-duplicate pairs logged: {len(duplicates)}")

# SAVE CSV
import csv

with open("bounding_box_duplicates.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["box1", "box2", "image_id1", "image_id2", "similarity"])
    for row in duplicates:
        writer.writerow(row)

print("CSV saved: bounding_box_duplicates.csv")

# FULL IMAGE SIM CHECKER

In [ ]:
# CONFIG
ZIP_PATH = "MyFigureCollection.zip"
XML_PATH = "annotations.xml"
SIM_THRESHOLD = 0.99
BATCH_SIZE = 128
K_NEIGHBORS = 10

EMB_FILE = "clip_embeddings_full.npy"

# LOAD TORCH/CLIP
import clip, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"USING DEVICE: {DEVICE}")
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()


# PARSE CVAT XML
import xml.etree.ElementTree as ET

tree = ET.parse(XML_PATH)
root = tree.getroot()

# Only include images with at least one bounding box
image_id_map = {}
for image in root.findall("image"):
    img_id = image.attrib["id"]
    img_name = image.attrib["name"]
    if image.findall("box"):
        image_id_map[img_name] = img_id

print(f"Images with at least one bounding box: {len(image_id_map)}")

# OPEN ZIP
import zipfile
zipf = zipfile.ZipFile(ZIP_PATH)

# Only keep images in XML with bounding boxes
image_files = [
    f for f in zipf.namelist()
    if f in image_id_map and f.lower().endswith((".png", ".jpg", ".jpeg"))
]

print(f"Total images with bounding boxes found in zip: {len(image_files)}")

# COMPUTE OR LOAD EMBEDDINGS
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
import gc
import os

image_keys = []

# LOAD EMBEDDINGS IF AVAILABLE
if os.path.exists(EMB_FILE):
    embeddings = np.load(EMB_FILE)
    print(f"Loaded embeddings: {embeddings.shape}")
else:
    embeddings_list = []

# GENERATE KEYS AND COMPUTE MISSING EMBEDDINGS
for i in tqdm(range(0, len(image_files), BATCH_SIZE), desc="Processing images"):
    batch_files = image_files[i:i+BATCH_SIZE]
    batch_tensors = []

    for img_path in batch_files:
        image_keys.append(img_path)  # always regenerate keys
        if not os.path.exists(EMB_FILE):  # compute embeddings only if not loaded
            try:
                with zipf.open(img_path) as f:
                    img = Image.open(f).convert("RGB")
                    batch_tensors.append(preprocess(img).unsqueeze(0))
            except Exception as e:
                print(f"Error loading {img_path}: {e}")

    if batch_tensors and not os.path.exists(EMB_FILE):
        batch_tensors = torch.cat(batch_tensors).to(DEVICE)
        with torch.no_grad():
            batch_emb = model.encode_image(batch_tensors)
            batch_emb = batch_emb / batch_emb.norm(dim=-1, keepdim=True)
            embeddings_list.append(batch_emb.cpu().numpy())

        del batch_tensors, batch_emb
        torch.cuda.empty_cache()
        gc.collect()

# SAVE EMBEDDINGS IF COMPUTED
if not os.path.exists(EMB_FILE):
    embeddings = np.vstack(embeddings_list).astype("float32")
    np.save(EMB_FILE, embeddings)
    print(f"Saved embeddings: {embeddings.shape}")

# BUILD FAISS INDEX
import faiss

faiss.normalize_L2(embeddings)
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

print("Faiss index built.")

# DEDUPLICATION
duplicates = []  # (image1, image2, image_id1, image_id2, similarity)
seen_pairs = set()

D, I = index.search(embeddings, K_NEIGHBORS)

for i, neighbors in enumerate(tqdm(I, desc="Finding duplicates")):
    anchor = image_keys[i]
    anchor_id = image_id_map.get(anchor, "N/A")

    for j_idx, sim in zip(neighbors[1:], D[i, 1:]):  # skip self
        neighbor = image_keys[j_idx]
        neighbor_id = image_id_map.get(neighbor, "N/A")

        if sim >= SIM_THRESHOLD:
            pair = tuple(sorted([anchor, neighbor]))
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                duplicates.append((anchor, neighbor, anchor_id, neighbor_id, float(sim)))

print(f"Total near-duplicate full-image pairs: {len(duplicates)}")

# SAVE CSV
import csv

with open("image_duplicates.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image1", "image2", "image_id1", "image_id2", "similarity"])
    writer.writerows(duplicates)

print("CSV saved: image_duplicates.csv")